In [1]:
# CC1 — Réseaux de neurones sur le dataset Iris
**Scikit-learn (`MLPClassifier`) vs TensorFlow/Keras**

## Introduction
Dans ce notebook, on résout **le même problème de classification** avec deux frameworks :
1. `MLPClassifier` de **Scikit-learn** ;
2. un réseau `Sequential` de **TensorFlow/Keras**.

SyntaxError: invalid syntax (2979925714.py, line 2)

In [ ]:
## Dataset Iris
# --- Bibliothèques générales ---
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, log_loss)

# Pour obtenir des résultats reproductibles
SEED = 42
np.random.seed(SEED)
### Chargement et exploration des données
# Chargement du dataset Iris (intégré à scikit-learn)
iris = load_iris()

# Transformation en DataFrame pour un affichage plus lisible
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

print("Premières lignes du dataset :")
display(df.head())

print("Dimensions du dataset (lignes, colonnes) :", df.shape)
print("Noms des variables :", iris.feature_names)
print("Classes :", list(iris.target_names))
print("Nombre d'exemples par classe :")
print(df['target'].value_counts().sort_index())
# Séparation X / y
X = iris.data
y = iris.target

# Division train / test (80 % / 20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print("Train :", X_train.shape, " Test :", X_test.shape)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform sur le train
X_test_scaled = scaler.transform(X_test)         # transform seulement sur le test
## Entraînement
mlp_clf = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=1000, random_state=SEED)

start = time.time()
mlp_clf.fit(X_train_scaled, y_train)
time_sk_iris = time.time() - start

print(f"Entraînement terminé en {time_sk_iris:.3f} s ({mlp_clf.n_iter_} itérations)")
## Prédiction et Accuracy
y_pred_sk = mlp_clf.predict(X_test_scaled)

acc_sk = accuracy_score(y_test, y_pred_sk)
print(f"Accuracy (test) MLPClassifier : {acc_sk:.4f}")
## Matrice de confusion et rapport de classification
cm_sk = confusion_matrix(y_test, y_pred_sk)
ConfusionMatrixDisplay(cm_sk, display_labels=iris.target_names).plot(cmap='Blues')
plt.title("Matrice de confusion — MLPClassifier (Scikit-learn)")
plt.show()

print("Classification report :")
print(classification_report(y_test, y_pred_sk, target_names=iris.target_names))
## Courbe de loss
plt.figure(figsize=(7, 4))
plt.plot(mlp_clf.loss_curve_)
plt.title("Courbe de loss — MLPClassifier")
plt.xlabel("Itérations")
plt.ylabel("Loss (entraînement)")
plt.grid(True)
plt.show()

print(f"Loss finale (entraînement) : {mlp_clf.loss_curve_[-1]:.4f}")

In [ ]:
# Partie 2 : TensorFlow/Keras — Classification Iris
## Préparation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# Reproductibilité
tf.keras.utils.set_random_seed(SEED)

print("Version de TensorFlow :", tf.__version__)
print("Données identiques à la partie 1 :", X_train_scaled.shape, X_test_scaled.shape)
## Construction du réseau
model_iris = Sequential([
    Input(shape=(4,)),                 # 4 variables d'entrée
    Dense(16, activation='relu'),      # 1re couche cachée
    Dense(8, activation='relu'),       # 2e couche cachée
    Dense(3, activation='softmax')     # sortie : 3 classes
])

model_iris.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',   # classes codées en entiers 0, 1, 2
    metrics=['accuracy']
)

model_iris.summary()

start = time.time()
history_iris = model_iris.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0          # mettre 1 pour voir la progression
)
time_keras_iris = time.time() - start
print(f"Entraînement terminé en {time_keras_iris:.2f} s")

# Évaluation sur le jeu de test
test_loss_keras, test_acc_keras = model_iris.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Loss (test)     : {test_loss_keras:.4f}")
print(f"Accuracy (test) : {test_acc_keras:.4f}")

# Prédictions : le modèle renvoie 3 probabilités par fleur -> on prend la plus grande
proba_keras = model_iris.predict(X_test_scaled, verbose=0)
y_pred_keras = np.argmax(proba_keras, axis=1)

acc_keras = accuracy_score(y_test, y_pred_keras)
print(f"Accuracy recalculée avec scikit-learn : {acc_keras:.4f}")

cm_keras = confusion_matrix(y_test, y_pred_keras)
ConfusionMatrixDisplay(cm_keras, display_labels=iris.target_names).plot(cmap='Greens')
plt.title("Matrice de confusion — Keras")
plt.show()

print(classification_report(y_test, y_pred_keras, target_names=iris.target_names))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_iris.history['loss'], label='Entraînement')
axes[0].plot(history_iris.history['val_loss'], label='Validation')
axes[0].set_title("Loss"); axes[0].set_xlabel("Epochs"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history_iris.history['accuracy'], label='Entraînement')
axes[1].plot(history_iris.history['val_accuracy'], label='Validation')
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epochs"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

loss_sk_test = log_loss(y_test, mlp_clf.predict_proba(X_test_scaled))
loss_keras_test = log_loss(y_test, proba_keras)

n_params_sk = sum(w.size for w in mlp_clf.coefs_) + sum(b.size for b in mlp_clf.intercepts_)

comparison_iris = pd.DataFrame({
    'Modèle': ['MLPClassifier', 'Neural Network'],
    'Framework': ['Scikit-learn', 'TensorFlow/Keras'],
    'Accuracy': [acc_sk, acc_keras],
    'Loss (test)': [loss_sk_test, loss_keras_test],
    'Temps entraînement (s)': [time_sk_iris, time_keras_iris],
    'Architecture': ['4-10-10-3 (ReLU)', '4-16-8-3 (ReLU + Softmax)'],
    'Nb paramètres': [n_params_sk, model_iris.count_params()],
})
display(comparison_iris.round(4))

# Graphique comparatif des accuracies
plt.figure(figsize=(5, 3.5))
plt.bar(['Scikit-learn', 'Keras'], [acc_sk, acc_keras], color=['steelblue', 'seagreen'])
plt.ylim(0, 1.05); plt.ylabel("Accuracy (test)"); plt.title("Accuracy : Sklearn vs Keras")
plt.show()

# Verdict automatique (calculé, pas écrit à la main)
if acc_sk > acc_keras:
    print("➜ Meilleure accuracy : MLPClassifier (Scikit-learn)")
elif acc_keras > acc_sk:
    print("➜ Meilleure accuracy : Neural Network (Keras)")
else:
    print("➜ Accuracy identique pour les deux modèles ; on départage avec la loss :",
          "Scikit-learn" if loss_sk_test < loss_keras_test else "Keras", "a la plus faible loss.")